# Nemotron EN–VI Guard — Vast.ai runner

Notebook này cố ý tách **profile → mini smoke → full run → inference**. Mọi launcher bên dưới stream thanh tiến độ trực tiếp. `ALLOW_FULL_RUN` mặc định là `False`.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time

if Path('/workspace/safety-dataset').exists():
    ROOT = Path('/workspace/safety-dataset')
elif (Path.cwd() / 'configs/phase0_experiments.json').exists():
    ROOT = Path.cwd().resolve()
else:
    ROOT = Path.cwd().resolve().parent
os.chdir(ROOT)
PYTHON = str(ROOT / '.venv/bin/python') if (ROOT / '.venv/bin/python').exists() else sys.executable
CONFIG = ROOT / 'configs/phase0_experiments.json'
RUNS = json.loads(CONFIG.read_text(encoding='utf-8'))['runs']
print('root=', ROOT)
print('python=', PYTHON)
subprocess.run(['nvidia-smi'], check=True)
print([run['id'] for run in RUNS])

## 1. Data preflight — phải pass trước khi GPU chạy

In [ ]:
subprocess.run([PYTHON, 'scripts/preflight_phase0_experiments.py'], check=True)

## 2. Profile mmBERT LoRA theo context — dừng riêng khi OOM

In [ ]:
PROFILE_BATCH = 1
profile_out = ROOT / f'reports/remote_profile/mmbert_fixed_b{PROFILE_BATCH}.json'
subprocess.run([
    PYTHON, 'scripts/benchmark_mmbert_lora_context.py',
    '--lengths', '512,1024,2048,4096,8192',
    '--batch-size', str(PROFILE_BATCH), '--repeats', '2',
    '--gradient-checkpointing', '--output', str(profile_out),
], check=True)
display(json.loads(profile_out.read_text(encoding='utf-8')))
schema_profile_out = ROOT / f'reports/remote_profile/mmbert_schema_b{PROFILE_BATCH}.json'
subprocess.run([
    PYTHON, 'scripts/benchmark_mmbert_schema_context.py',
    '--lengths', '512,1024,2048,4096,8192',
    '--batch-size', str(PROFILE_BATCH), '--repeats', '2',
    '--gradient-checkpointing', '--output', str(schema_profile_out),
], check=True)
display(json.loads(schema_profile_out.read_text(encoding='utf-8')))

## 3. Chọn run và ngân sách

Giữ `MAX_STEPS=50` cho mini smoke. Chỉ đặt `MAX_STEPS=0` và `ALLOW_FULL_RUN=True` sau khi profile, checkpoint reload và inference mini đều pass.

In [ ]:
RUN_ID = 'E2-M-EV-GLI-COMPAT-8K'  # E1, E2, E3, E4, E5 hoặc E7
MICRO_BATCH = 1
EPOCHS = 2
MAX_STEPS = 50
PRECISION = 'auto'  # Turing -> fp16; Ampere/Blackwell -> bf16 when supported
ALLOW_FULL_RUN = False

if MAX_STEPS == 0 and not ALLOW_FULL_RUN:
    raise RuntimeError('Full run đang khóa. Hãy profile + mini smoke trước.')
run = next(item for item in RUNS if item['id'] == RUN_ID)
display(run)

## 4. Train — tqdm/log stream hiện ngay trong notebook

In [ ]:
if run['model_kind'] == 'gliguard_schema':
    cmd = [PYTHON, 'scripts/train_gliguard_experiment.py', '--run-id', RUN_ID,
           '--micro-batch-size', str(MICRO_BATCH), '--epochs', str(EPOCHS),
           '--precision', PRECISION]
    if MAX_STEPS:
        cmd += ['--max-optimizer-steps', str(MAX_STEPS)]
elif run['model_kind'] == 'mmbert_fixed':
    cmd = [PYTHON, 'scripts/train_mmbert_fixed_experiment.py', '--run-id', RUN_ID,
           '--micro-batch-size', str(MICRO_BATCH), '--epochs', str(EPOCHS),
           '--precision', PRECISION]
    if MAX_STEPS:
        cmd += ['--max-optimizer-steps', str(MAX_STEPS)]
elif run['model_kind'] == 'mmbert_schema':
    cmd = [PYTHON, 'scripts/train_mmbert_schema_experiment.py', '--run-id', RUN_ID,
           '--micro-batch-size', str(MICRO_BATCH), '--epochs', str(EPOCHS),
           '--precision', PRECISION]
    if MAX_STEPS:
        cmd += ['--max-optimizer-steps', str(MAX_STEPS)]
else:
    raise ValueError(run['model_kind'])
print(' '.join(cmd))
subprocess.run(cmd, check=True)

## 5. Inference độc lập và lưu predictions

Cell này dành cho mmBERT. Có thể đổi manifest sang Nemotron test, SEA native hoặc SEA shared-truncated. Với E7 hãy chạy cả canonical và reversed.

In [ ]:
if run['model_kind'] in {'mmbert_fixed', 'mmbert_schema'}:
    EVAL_MANIFEST = Path('data/eval_shared_gliguard_512/sea_paired_native.jsonl')
    LABEL_ORDER = 'reversed' if run['model_kind'] == 'mmbert_schema' else 'canonical'
    final_dir = ROOT / 'reports/experiment_runs' / RUN_ID / 'final'
    eval_dir = ROOT / 'reports/experiment_runs' / RUN_ID / f'eval_{EVAL_MANIFEST.stem}_{LABEL_ORDER}'
    subprocess.run([
        PYTHON, 'scripts/evaluate_scalable_mmbert_checkpoint.py',
        '--model-kind', 'schema' if run['model_kind'] == 'mmbert_schema' else 'fixed',
        '--checkpoint', str(final_dir), '--base-model', run['base_model'],
        '--manifest', str(EVAL_MANIFEST), '--output-dir', str(eval_dir),
        '--max-length', str(run['context']), '--batch-size', str(MICRO_BATCH),
        '--precision', PRECISION, '--label-order', LABEL_ORDER,
    ], check=True)
    display(json.loads((eval_dir / 'metrics.json').read_text(encoding='utf-8')))
else:
    EVAL_MANIFEST = Path('data/eval_shared_gliguard_512/sea_paired_native.jsonl')
    adapter = ROOT / 'reports/experiment_runs' / RUN_ID / 'final'
    eval_dir = ROOT / 'reports/experiment_runs' / RUN_ID / f'eval_{EVAL_MANIFEST.stem}'
    subprocess.run([
        PYTHON, 'scripts/evaluate_gliguard_checkpoint.py',
        '--base-model', run['base_model'], '--adapter', str(adapter),
        '--manifest', str(EVAL_MANIFEST), '--output-dir', str(eval_dir),
        '--batch-size', str(max(1, MICRO_BATCH)),
    ], check=True)
    display(json.loads((eval_dir / 'metrics.json').read_text(encoding='utf-8')))

## 6. Chạy toàn bộ ma trận đánh giá đã khóa

Đặt `MATRIX_LIMIT=8` để smoke; đặt `None` cho đánh giá đầy đủ. E7 tự chạy cả canonical và reversed schema.

In [ ]:
MATRIX_LIMIT = 8
matrix_cmd = [PYTHON, 'scripts/evaluate_experiment_matrix.py', '--run-id', RUN_ID,
              '--batch-size', str(MICRO_BATCH), '--precision', PRECISION]
if MATRIX_LIMIT is not None:
    matrix_cmd += ['--limit', str(MATRIX_LIMIT)]
subprocess.run(matrix_cmd, check=True)

## 7. Xem nhanh trạng thái/checkpoint hiện có

In [ ]:
run_root = ROOT / 'reports/experiment_runs'
for metrics_path in sorted(run_root.glob('*/metrics.json')):
    payload = json.loads(metrics_path.read_text(encoding='utf-8'))
    print(metrics_path.parent.name, payload.get('status'), payload.get('completed_optimizer_steps', payload.get('train_result', {}).get('global_step')))
for checkpoint in sorted(run_root.glob('*/checkpoints/step-*')):
    print('checkpoint', checkpoint)